# Evaluating Chunking Methods

This notebook evaluates how three different chunking methods (fixed-size, markdown-based, and LLM-semantic) affects performance in a baking assistant RAG system. The evaluation uses batch replay test prompts—multi-turn conversations that include both the ideal model response and the ideal chunk(s) that should be retrieved.

## Which Metrics were used:
There are two ways we are measuring the chunks affect on LLM's RAG performance:
A) Quatitative Metrics (Token-Level Chunk Comparison):
Since the goal is to measure chunking quality, we compare for each turn in the conversation the chunks that were returned by the AI system and the ideal chunk that we would have wanted. Therefore, the all metrics are token-level comparison between the retrieved chunks vs. ideal chunks:

* Precision: proportion of retrieved tokens that are correct.

* Recall: proportion of gold tokens successfully retrieved

* F1 score: harmonic mean of precision and recall.

* ROUGE-L: longest common subsequence overlap between retrieved and ideal chunk text. (Sequence-based similarity metric)

* IoU (token overlap): overall overlap between retrieved and gold tokens

These metrics directly measure how well each chunking method supports accurate retrieval.

#### Definitions

- $t_e$: set of gold (highlight) tokens  
- $t_r$: set of retrieved tokens  
- $q$: a specific query  
- $\mathbf{C}$: the chunked corpus (document split into chunks)

$\text{Precision}_q(\mathbf{C}) = \frac{|t_e \cap t_r|}{|t_r|}$

$\text{Recall}_q(\mathbf{C}) = \frac{|t_e \cap t_r|}{|t_e|}$

$F1_q(\mathbf{C}) = 2 \cdot \frac{\text{Precision}_q(\mathbf{C}) \cdot \text{Recall}_q(\mathbf{C})}{\text{Precision}_q(\mathbf{C}) + \text{Recall}_q(\mathbf{C})}$

###### Longest Common Subsequence (LCS) based similarity metric.

$\text{ROUGE-L}_q(\mathbf{C}) = \frac{\text{LCS}(t_e, t_r)}{\max(|t_e|, |t_r|)}$

$\text{IoU}_q(\mathbf{C}) = \frac{|t_e \cap t_r|}{|t_e \cup t_r|}$

B) Qualitative Metrics: 
To understand whether chunking affects downstream model performance, we evaluate the assistant’s final answers using an LLM-as-a-Judge. This step matters because poor chunking does not always translate to poor answers, and vice versa. However, the system prompt forced the model to rely only on retrieved knowledge (except for greetings). This isolates the effect of chunking. 

In real user testing, the model was allowed to fall back on internal knowledge with appropriate disclaimers.

The LLM-as-a-Judge checked for 3 different metrics on a score from 1-5:
* Groundedness (Chunk Alignment): How much does the answer reflect the ideal chunk? The LLM judge should use just the ideal chunk and the baking LLM response to evaluate this. 

* Factual Accuracy: Does the answer preserve correct information? e.g. correct measurements or oven temperatures. The LLM judge should use just the ideal response and the baking LLM response to evaluate this. 

* Helpfulness: Is the answer still useful to the user, given the restricted context? The LLM judge should use the users query and previous dialogue history to determine if the baking LLM's response was helpful.  

Therefore, the LLM-as-a-Judge has access to the current conversation history, the ideal chunk, the ideal response, and LLM's actual response, and the User query.

## Why Two Test Batches?
1) Batch 1 — Natural Conversation (called Batch_test_1_mix)
This batch simulates realistic user interaction, including greetings, off-topic questions, and mistakes. Some turns have no ideal chunk (NA) because the question is off-topic or a greeting. This is important to evaluate how the model behaves in realistic user interaction (E.g. if the question is not relevant does it correctly re-direct the user?). 
It checks: 
* Conversational appropriateness
* Observe retrieval quality on mixed question types
* Ensure that irrelevant turns do not produce spurious retrieval
* Check if context is carried on in multi-turn conversation (does the AI remember and use all the information it was given to find the best chunk)

2) Batch 2 — Controlled Chunking Evaluation (called Batch_test_2_specific)

This batch is designed specifically to test chunk retrieval and specifically how performance varies across question types. There are 10 multi-turn dialogues were each 2 dialogues focus only around questions of a similar category. Every turn has a known ideal chunk.

The four categories are :

A) Recipe Questions — long, multi-section answers e.g. “How can I make chocolate chip cookies?”

B) Modification / Troubleshooting Questions — short, targeted fixes e.g “How can I make this gluten-free?”

C) Technical / Technique Questions — medium-length procedural skills e.g. “How do I temper chocolate?”

D) Conceptual / Scientific Questions — explanations of underlying principles e.g. “Explain the Maillard reaction.”

### Summary:
This evaluation setup lets us measure:

* How effectively each chunking method supports retrieval (quantitative)

* How chunking quality influences the model’s final answers (qualitative)

* How performance varies across question types

Together, these results indicate which chunking strategy is most suitable for a baking-themed RAG system.


# Quatitative Evaluation 

## Setup and Chunking Method

In [ ]:
#Necessary Imports:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import gradio as gr
from dotenv import load_dotenv
import json  
from datetime import datetime 
import os 
import json
from rouge_score import rouge_scorer
from collections import defaultdict
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


#Loading google's api keya again from .env file
load_dotenv() 
DATA_PATH = r"data"
CHROMA_PATH = r"chroma_db"

#Loading the same embedding model used to create the chroma databases
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small"   
)

In [2]:
#Setting up the chroma database
#3 collections:
#cooking_chunks_fixed
#cooking_chunks_markdown
#cooking_chunks_semantic

#Run different collection names according to the one being tested
#Chunk_method ="cooking_chunks_fixed"
#Chunk_method ="cooking_chunks_markdown"
#Chunk_method ="cooking_chunks_semantic"

COLLECTION_NAME = "cooking_chunks_fixed"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_model,   
    persist_directory=CHROMA_PATH
)


In [3]:
########################### Defines all the helper methods needed to Evaluate ############


#This method retrieves the actual chunk bsaed on the id:
def get_chunk_text_by_id(chunk_id):
    """Retrieve actual chunk text from Chroma DB given the chunk_id."""
    res = vector_store.get(where={"chunk_id": int(chunk_id)})
    if res and res["documents"]:
        return res["documents"][0]
    return ""

#This method tokenises the input to allow token level comparison
def tokenize(text):
    return text.lower().replace("–", "-").replace("—", "-").split()


#This method calculate for a single golden chunk and the single retrieved chunk
##IOU
##F1
##Recall
##Precision
def token_overlap_metrics(gold, chunk):
    gold_tokens = set(tokenize(gold))
    chunk_tokens = set(tokenize(chunk))

    intersection = gold_tokens & chunk_tokens
    union = gold_tokens | chunk_tokens

    precision = len(intersection) / len(chunk_tokens) if chunk_tokens else 0
    recall = len(intersection) / len(gold_tokens) if gold_tokens else 0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0
    iou = len(intersection) / len(union) if union else 0

    return precision, recall, f1, iou

#Defining Rouger_Scorere from the RougeScorer library 
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

#The method returns the rouge_l for the gold/ideal chunk and the single retrieved chunk
def rouge_l_score(gold, chunk):
    """Return ROUGE-L F1 score."""
    score = scorer.score(gold.lower(), chunk.lower())
    return score["rougeL"].fmeasure




In [71]:
#This method takes in a JSON test file and outputs the metrics for each turn in the conversation
#How it deals with turns that don't have an ideal CHunk? 
# ->these ideal chunks are labelled NA and the method will skip them (not count them)

#How it deals with many chunks being retrieved for a single query (fixed size of 4 chunks)
# ->Finds the metrics for each retrieved chunk and gold chunk, and takes the single maximum values

#How it deal with the gold/ideal chunk being more than one?
# ->Finds the maximum values for both of those ideal chunks and averages them (measures if the whole system retrieved the best chunk)

#Note: slight bias as for recipe chunks, whole recipes take too much space to be one ideal chunk, so the ideal chunk for that was broken down into the recipies parts and treated as multiple chunks

def evaluate_from_json(json_path):
    """Runs Best-Match Multi-Gold evaluation and returns per-turn scores.
    Input: JSON multi-turn conversation test file (clearly labeled golden chunk and ideal response, see example file)
    Output: A JSON like format file with the metrics evaluated for each single turn in the conversation """

    #Open the JSON file
    with open(json_path, "r") as f:
        data = json.load(f)

    results = []

    #For each dialogue (dialog_idx) in the set of dialogues
    for dialog_idx, dialogue in enumerate(data, start=1):

        #Dataset sometimes stores turns as a flat list instead of {"turns": [...]}
        if isinstance(dialogue, dict) and "turns" in dialogue:
            turns = dialogue["turns"]
        else:
            turns = dialogue  #Already a list of turn dicts

            #for each turn within a dialogue
        for turn in turns:
            # Extract gold chunk(s)
            raw_gold = turn.get("ideal_chunk", "")

            #Handle NA or empty cases
            if isinstance(raw_gold, str):
                cleaned = raw_gold.strip()
                if cleaned == "" or cleaned.upper() == "NA":
                    continue
                gold_chunks = [cleaned]

            elif isinstance(raw_gold, list):
                gold_chunks = [g.strip() for g in raw_gold if g.strip() != ""]
                if len(gold_chunks) == 0:
                    continue
            else:
                continue
    
            #Retrieve the string of the chunks retrieved from Chroma
            retrieved_ids = [item["chunk_id"] for item in turn.get("retrieved_context", [])]

            retrieved_chunks = [
                get_chunk_text_by_id(cid)
                for cid in retrieved_ids
            ]

            #Compute Best-Match scores
            precision_scores = []
            recall_scores = []
            f1_scores = []
            iou_scores = []
            rougeL_scores = []

            #In cases were gold chunk is more than one
            for gold in gold_chunks:

                max_p = max_r = max_f = max_iou = max_rl = 0

                #Compare each retrieved chunk to one of the golden ones
                for chunk in retrieved_chunks:
                    #print("new chunk")
                    #print(chunk)
                    p, r, f1, iou = token_overlap_metrics(gold, chunk)
                    rl = rouge_l_score(gold, chunk)

                    max_p = max(max_p, p)
                    max_r = max(max_r, r)
                    max_f = max(max_f, f1)
                    max_iou = max(max_iou, iou)
                    max_rl = max(max_rl, rl)

                #Store best score for this gold
                precision_scores.append(max_p)
                recall_scores.append(max_r)
                f1_scores.append(max_f)
                iou_scores.append(max_iou)
                rougeL_scores.append(max_rl)

            #Average across gold chunks
            avg_precision = sum(precision_scores) / len(precision_scores)
            avg_recall    = sum(recall_scores)    / len(recall_scores)
            avg_f1        = sum(f1_scores)        / len(f1_scores)
            avg_iou       = sum(iou_scores)       / len(iou_scores)
            avg_rougeL    = sum(rougeL_scores)    / len(rougeL_scores)

            #Append per-turn result
            results.append({
                "dialogue_id": dialog_idx,
                "precision": avg_precision,
                "recall": avg_recall,
                "f1": avg_f1,
                "iou": avg_iou,
                "rouge_l": avg_rougeL
            })

    return results


In [73]:
#To test
result_for_each_turn = evaluate_from_json("Test_2_Fixed_results.json")
#See that the metrics will be returned for every turn in the conversation
result_for_each_turn

[{'dialogue_id': 1,
  'precision': 0.33088235294117646,
  'recall': 0.627906976744186,
  'f1': 0.43243243243243246,
  'iou': 0.305,
  'rouge_l': 0.38591852722287506},
 {'dialogue_id': 1,
  'precision': 0.5147058823529411,
  'recall': 0.9722222222222222,
  'f1': 0.6730769230769229,
  'iou': 0.5072463768115942,
  'rouge_l': 0.6805555555555556},
 {'dialogue_id': 1,
  'precision': 0.26229508196721313,
  'recall': 0.14035087719298245,
  'f1': 0.18285714285714283,
  'iou': 0.10062893081761007,
  'rouge_l': 0.1596958174904943},
 {'dialogue_id': 1,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'iou': 1.0,
  'rouge_l': 1.0},
 {'dialogue_id': 2,
  'precision': 0.6934865900383143,
  'recall': 0.7666666666666666,
  'f1': 0.6215078453884423,
  'iou': 0.46015325670498086,
  'rouge_l': 0.5649681022438328},
 {'dialogue_id': 2,
  'precision': 0.759493670886076,
  'recall': 1.0,
  'f1': 0.8633093525179857,
  'iou': 0.759493670886076,
  'rouge_l': 0.854054054054054},
 {'dialogue_id': 2,
  'precisio

In [74]:
#Saving the results
output_file = "metrics_for_every_single_turn.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_for_each_turn, f, indent=4)

In [75]:
#This Method takes the output of the previous cell/ JSON file containing the single turns
#and average per dialogue what the metrics are
#Outputs the average for each dialogue (divides each dialgoue by turns)


#Why not do this directly?
#->The earlier step is a good checker to debug; can clearly see the scores for each turn and inspect
#->Once you averaged the scores it difficult to see if the code is correctly calcualting the metrics and
#->Correctly skipped the NA dialogues 

def average_per_dialogue(results):
    """Compute averages of metrics for each dialogue.
    Input: JSON format of each metric for each turn
    Output: JSON format of each dialgoues average metrics"""
    grouped = defaultdict(list)

    # group all turns by dialogue ID
    for r in results:
        grouped[r["dialogue_id"]].append(r)

    dialogue_avgs = {}

    for d_id, turns in grouped.items():
        avg_precision = sum(t["precision"] for t in turns) / len(turns)
        avg_recall    = sum(t["recall"]    for t in turns) / len(turns)
        avg_f1        = sum(t["f1"]        for t in turns) / len(turns)
        avg_iou       = sum(t["iou"]       for t in turns) / len(turns)
        avg_rl        = sum(t["rouge_l"]   for t in turns) / len(turns)

        dialogue_avgs[d_id] = {
            "precision": avg_precision,
            "recall": avg_recall,
            "f1": avg_f1,
            "iou": avg_iou,
            "rouge_l": avg_rl
        }

    return dialogue_avgs


In [76]:
result_for_each_dialogue = average_per_dialogue(result_for_each_turn)
result_for_each_dialogue

{1: {'precision': 0.5269708293153327,
  'recall': 0.6851200190398476,
  'f1': 0.5720916245916245,
  'iou': 0.47821882690730105,
  'rouge_l': 0.5565424750672312},
 2: {'precision': 0.5940313949916743,
  'recall': 0.6444444444444445,
  'f1': 0.5686979312170363,
  'iou': 0.448016280488885,
  'rouge_l': 0.5227851632104067},
 3: {'precision': 0.657000358037952,
  'recall': 0.9869047619047618,
  'f1': 0.7519449114722793,
  'iou': 0.6445507456348513,
  'rouge_l': 0.7223872499436194},
 4: {'precision': 0.9455762987012987,
  'recall': 0.8212113079262838,
  'f1': 0.8778005610088685,
  'iou': 0.7836445802194262,
  'rouge_l': 0.8008537191361796},
 5: {'precision': 0.7332777341705914,
  'recall': 1.0,
  'f1': 0.8302578773099129,
  'iou': 0.7332777341705914,
  'rouge_l': 0.8255351331222747},
 6: {'precision': 0.6341298825023818,
  'recall': 0.9333333333333333,
  'f1': 0.747835497835498,
  'iou': 0.6154259338910082,
  'rouge_l': 0.6947271534942323},
 7: {'precision': 0.6435290404040404,
  'recall': 1

In [77]:
#Saving the results
output_file = "metrics_for_each_dialogue.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_for_each_dialogue, f, indent=4)

In [78]:
#This last method is the one that is most important
#The other two are still necessary for debugging and checking correct averaging
#This is what determines the overall average for that batch-replay script

#So having the two previous method is like a forced check that the code is correct
#IMPORTANT: the input here is the output of evaluate_from_json!!!! the other method average_per_dialogue is more of a checker function

def aggregate_overall_metrics(results):
    """Compute overall averages across all scored turns.
    Input: JSON file with metric results for each turn 
    Output: JSON file with average metrics for the whole batch replay script"""

    if not results: #in case it's empty/has no data
        print("ERROR wrong data type or no data is contained in the results")
        return None

    total = len(results)

    avg_precision = sum(r["precision"] for r in results) / total
    avg_recall    = sum(r["recall"]    for r in results) / total
    avg_f1        = sum(r["f1"]        for r in results) / total
    avg_iou       = sum(r["iou"]       for r in results) / total
    avg_rouge_l   = sum(r["rouge_l"]   for r in results) / total

    return {
        "total_turns_scored": total,
        "avg_precision": avg_precision,
        "avg_recall": avg_recall,
        "avg_f1": avg_f1,
        "avg_iou": avg_iou,
        "avg_rouge_l": avg_rouge_l
    }


In [79]:
results_overall = aggregate_overall_metrics(result_for_each_turn)
results_overall 

{'total_turns_scored': 32,
 'avg_precision': 0.668053877998236,
 'avg_recall': 0.9001657648790224,
 'avg_f1': 0.7371760088306002,
 'avg_iou': 0.6288066165905138,
 'avg_rouge_l': 0.7053301053444895}

In [80]:
#Saving the results
output_file = "metrics_across_everything.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results_overall, f, indent=4)

# Qualitative Metrics (LLM as a Judge)

In [158]:
#For LLM as a Judge these are the extra imports needed:
#Note it is important that the Judge LLM is more powerful and slightly different then the actually LLM
#Only this way can it accurately judge the response
from openai import OpenAI
#Loading api key from .env

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [159]:
#First we need to create the system prompt for the LLM judge

#The judge will have acess to the history of the conversation, the users query, the LLM response, the ideal response, and the ideal chunk
def build_judge_prompt(history, user, response, ground_truth, ideal_chunk):
    """
    Builds the LLM-as-a-Judge prompt to evaluate an assistant response using:
    - Factual Accuracy
    - Helpfulness
    - Groundedness (if an ideal chunk exists)
    """

    #Converts dialogue history to readable text for the LLM judge
    history_text = ""
    for h in history:
        history_text += f"{h['role'].capitalize()}: {h['content']}\n"

    #Base prompt that applies to all cases
    base_prompt = f"""
You are an impartial evaluation model. Your task is to evaluate the assistant’s answer 
using **three metrics**. You MUST follow the comparison rules for each metric exactly.

The domain of this system is *baking*. 
If the question is not about baking (e.g., greetings or mistakes), evaluate the metrics 
logically given the information provided.

METRIC DEFINITIONS AND REQUIRED REFERENCES:
1. **Factual Accuracy (1–5):**
   Compare ONLY:
       - the assistant's response
       - the ground-truth answer
   Rate how correct, precise, and non-hallucinatory the assistant's answer is.
   Ignore the ideal chunk when scoring accuracy. Did the response use the exact same value and metrics as the ground-truth answer?

2. **Helpfulness (1–5):**
   Compare ONLY:
       - the user’s question
       - the assistant’s response
   Rate how useful, actionable, and responsive the answer is to the user’s intent.
   An answer can be factually correct but still unhelpful, or vice versa.

3. **Groundedness (1–5 or "N/A"):**
   ONLY evaluate if an ideal chunk is provided.
   Compare ONLY:
       - the assistant's response
       - the ideal knowledge chunk
   Rate how well the answer is supported by that chunk.
   If no chunk exists (ideal_chunk == "NA"), groundedness must be "N/A".


Return ONLY a JSON object with these fields and numeric scores.
DIALOGUE HISTORY
{history_text}

CURRENT TURN
User question:
{user}

Assistant response:
{response}

Ground-truth answer (used ONLY for factual accuracy):
{ground_truth}
"""

    # If ideal chunk exists, evaluate groundedness normally
    if ideal_chunk and ideal_chunk.strip().upper() != "NA":
        base_prompt += f"""
Ideal (gold) chunk (used ONLY for groundedness):
{ideal_chunk}

Return JSON exactly in this format:
{{
 "factual_accuracy": <1-5>,
 "helpfulness": <1-5>,
 "groundedness": <1-5>
}}
"""
    #If no ideal chunk exists then can't evaluate groundesnes; groundedness = N/A
    else:
        base_prompt += """
No ideal chunk is provided for this turn.

Return JSON exactly in this format:
{
 "factual_accuracy": <1-5>,
 "helpfulness": <1-5>,
 "groundedness": "N/A"
}
"""

    return base_prompt


In [160]:
#This is the method that runs the LLM judge with the prompt given above

#The model used as the judge is specified as model
#This case we used gpt-4o-mini because it is stronger than gpt-5-nano but still cheap (limited credits)
#This evaluation file will be outputed as a json file inside the project folder

def judge_batch_file(json_path, output_path="LLM_judge_scores.json", model="gpt-4o-mini"):
    """
    Evaluates every turn in a batch replay JSON file using LLM-as-a-Judge.
    Input: JSON_path of where the batch replay script to evaluate is.
    Output: produces a JSON file with per-turn ratings.
    """

    #Open's the recorded test conversation JSON file (this get created in the Notebook chatbot_interface)
    with open(json_path, "r") as f:
        dialogues = json.load(f)

    #List to store all results
    all_results = []

    #for each dialogue within all the dialogues
    for dialogue in dialogues:
        history = []

        #for each turn within a single dialogue 
        for turn in dialogue["turns"]:

            #records information from the JSON file to give the LLM as a judge
            user = turn["user_utterance"]
            system_response = turn["system_response"]
            ground_truth = turn["ground_truth"]
            raw_ideal_chunk = turn.get("ideal_chunk", "NA")

            #These are the tests necessary for what gets retrived from ideal_chunk
            #In cases were the chunk does not exist:
            if raw_ideal_chunk == "NA" or raw_ideal_chunk is None:
                ideal_chunk = "NA"

            #In cases were the chunk exists as a string:
            elif isinstance(raw_ideal_chunk, str):
                ideal_chunk = raw_ideal_chunk

            #In cases were there are multiple chunks/list, then store them all mergwed
            elif isinstance(raw_ideal_chunk, list):
                ideal_chunk = "\n\n".join(raw_ideal_chunk)
            #Last case, if the format is something else/unexpect from the previous ones
            else:
                ideal_chunk = str(raw_ideal_chunk)

            #Builds judge prompt from before with that infromation
            prompt = build_judge_prompt(
                history=history,
                user=user,
                response=system_response,
                ground_truth=ground_truth,
                ideal_chunk=ideal_chunk
            )

            #Calls OPENAi model and passes it the prompt
            judge_output = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an impartial evaluation model."},
                    {"role": "user", "content": prompt}
                ]
            )

            raw_json = judge_output.choices[0].message.content

            #Gets scores returned from the Judge
            try:
                scores = json.loads(raw_json)
            except:
                print("ERROR PARSING JUDGE OUTPUT:", raw_json)
                continue

            #Saves those scores as a JSON file to add to the bigger list
            result = {
                "dialogue_id": dialogue["dialogue_id"],
                "user_utterance": user,
                "system_response": system_response,
                "ground_truth": ground_truth,
                "ideal_chunk": ideal_chunk,
                "scores": scores,
                "timestamp": datetime.now().isoformat() + "Z"
            }

            all_results.append(result)

            #Update history for next turn (just for that dialogue, then history get cleared for next dialogue, no contamination)
            history.append({"role": "user", "content": user})
            history.append({"role": "assistant", "content": system_response})

    #Save results
    with open(output_path, "w") as f:
        json.dump(all_results, f, indent=4)

    print(f"Saved judge scores to {output_path}")
    return all_results


In [161]:
results_llm = judge_batch_file("semantic_test_2_del.json")
results_llm

Saved judge scores to LLM_judge_scores.json


[{'dialogue_id': 1,
  'user_utterance': 'Do you know any cookie recipes? Just give me an overview of the ones you know?',
  'system_response': 'Sure — here’s what I have in my baking knowledge base about cookies:\n\nCookie recipe in the knowledge base\n- Chewy Almond Cookies\n  - 4 ingredients, oil-free, and naturally gluten-free almond cookies\n  - Ingredients:\n    - Almond flour\n    - Granulated sugar\n    - Eggs\n    - Natural almond extract\n    - A pinch of salt\n  - Instructions (summary of steps as listed):\n    1) Preheat oven to 350°F (325°F convection). Line 1–2 baking trays with parchment.\n    2) In a large bowl, mix together eggs, sugar, and almond extract.\n    3) Add almond flour and salt; mix with a spoon or spatula.\n    4) Form into 1 tbsp dough mounds, roll into balls, and place on baking sheet 1–2" apart. Decorate with a few sliced almonds.\n    5) Bake for 15–17 minutes, or until the tops are dry to the touch and the edges/bottom are golden brown.\n    6) Let coo

In [162]:
#Saving the results
output_file = "LLM_judge_full_output.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results_llm, f, indent=4)

In [163]:
df = pd.DataFrame(results_llm)
df

,dialogue_id,user_utterance,system_response,ground_truth,ideal_chunk,scores,timestamp
0,1,Do you know any cookie recipes? Just give me a...,Sure — here’s what I have in my baking knowled...,Yes! I have a few. You can try chewy chocolate...,# Recipe chocolate chip cookies\nDelicious che...,"{'factual_accuracy': 3, 'helpfulness': 4, 'gro...",2025-12-08T18:38:42.587187Z
1,1,What ingredients do I need for chocolate chip ...,Here are the ingredients for chocolate chip co...,Here are the ingredients for Delicious chewy c...,"""# Recipe chocolate chip cookies Ingredients\n...","{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:43.534049Z
2,1,What steps or the method needed to make them?,"From the knowledge base, the chocolate chip co...",Here are the steps: \nPreheat the oven to 190º...,# Recipe chocolate chip cookies Methods\nInstr...,"{'factual_accuracy': 3, 'helpfulness': 3, 'gro...",2025-12-08T18:38:44.429412Z
3,1,Are there any troubleshoots or notes I should ...,Here are the troubleshooting notes from the Co...,"Yes! When baking cookies, it helps to start ch...",# Recipe chocolate chip cookies Notes Troubles...,"{'factual_accuracy': 2, 'helpfulness': 3, 'gro...",2025-12-08T18:38:45.455050Z
4,2,"Give me the full recipe, step by step, to bake...",Here is the angel food cake recipe as provided...,Sure. Here are the ingredients for a perfect a...,# Angel Food Cake Recipe\nUsing only 6 ingredi...,"{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:46.785380Z
5,2,Actually I don’t think I have those ingredient...,Here are the ingredients for the Easy caramel ...,"Yes, ofcourse! These are the ingredients for t...",225g softened salted butter\nplus extra for th...,"{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:48.014249Z
6,2,"Great! I should have all those things, tell me...",Here are the Easy caramel cake ingredients and...,Here are the steps: \n1) Heat the oven to 180C...,## Easy caramel cake Method\n1) Heat the oven ...,"{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:49.138466Z
7,3,Give me the ingredients to a lemon drizzle cak...,Lemon drizzle cake ingredients:\n\n- 225g unsa...,Here are the ingredients for a lemon drizzle c...,# Recipe Lemon drizzle cake Ingredients\n225g ...,"{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:50.266012Z
8,3,Can I make this recipe gluten free? How?,Yes. To make this lemon drizzle cake gluten-fr...,"Yes, simply replace the self-raising flour wit...","To make this lemon drizzle cake gluten-free, s...","{'factual_accuracy': 5, 'helpfulness': 5, 'gro...",2025-12-08T18:38:52.753674Z
9,3,I don’t have eggs can I still make this recipe...,- For the lemon drizzle cake: Eggs are essenti...,Unfortunately no. Eggs play a crucial role as ...,Eggs play a crucial role in this lemon drizzle...,"{'factual_accuracy': 4, 'helpfulness': 5, 'gro...",2025-12-08T18:38:53.773821Z


In [164]:
scores_df = df["scores"].apply(pd.Series)
df_expanded = pd.concat([df.drop(columns=["scores"]), scores_df], axis=1)
df_expanded

,dialogue_id,user_utterance,system_response,ground_truth,ideal_chunk,timestamp,factual_accuracy,helpfulness,groundedness
0,1,Do you know any cookie recipes? Just give me a...,Sure — here’s what I have in my baking knowled...,Yes! I have a few. You can try chewy chocolate...,# Recipe chocolate chip cookies\nDelicious che...,2025-12-08T18:38:42.587187Z,3,4,2
1,1,What ingredients do I need for chocolate chip ...,Here are the ingredients for chocolate chip co...,Here are the ingredients for Delicious chewy c...,"""# Recipe chocolate chip cookies Ingredients\n...",2025-12-08T18:38:43.534049Z,5,5,5
2,1,What steps or the method needed to make them?,"From the knowledge base, the chocolate chip co...",Here are the steps: \nPreheat the oven to 190º...,# Recipe chocolate chip cookies Methods\nInstr...,2025-12-08T18:38:44.429412Z,3,3,3
3,1,Are there any troubleshoots or notes I should ...,Here are the troubleshooting notes from the Co...,"Yes! When baking cookies, it helps to start ch...",# Recipe chocolate chip cookies Notes Troubles...,2025-12-08T18:38:45.455050Z,2,3,2
4,2,"Give me the full recipe, step by step, to bake...",Here is the angel food cake recipe as provided...,Sure. Here are the ingredients for a perfect a...,# Angel Food Cake Recipe\nUsing only 6 ingredi...,2025-12-08T18:38:46.785380Z,5,5,5
5,2,Actually I don’t think I have those ingredient...,Here are the ingredients for the Easy caramel ...,"Yes, ofcourse! These are the ingredients for t...",225g softened salted butter\nplus extra for th...,2025-12-08T18:38:48.014249Z,5,5,5
6,2,"Great! I should have all those things, tell me...",Here are the Easy caramel cake ingredients and...,Here are the steps: \n1) Heat the oven to 180C...,## Easy caramel cake Method\n1) Heat the oven ...,2025-12-08T18:38:49.138466Z,5,5,5
7,3,Give me the ingredients to a lemon drizzle cak...,Lemon drizzle cake ingredients:\n\n- 225g unsa...,Here are the ingredients for a lemon drizzle c...,# Recipe Lemon drizzle cake Ingredients\n225g ...,2025-12-08T18:38:50.266012Z,5,5,5
8,3,Can I make this recipe gluten free? How?,Yes. To make this lemon drizzle cake gluten-fr...,"Yes, simply replace the self-raising flour wit...","To make this lemon drizzle cake gluten-free, s...",2025-12-08T18:38:52.753674Z,5,5,5
9,3,I don’t have eggs can I still make this recipe...,- For the lemon drizzle cake: Eggs are essenti...,Unfortunately no. Eggs play a crucial role as ...,Eggs play a crucial role in this lemon drizzle...,2025-12-08T18:38:53.773821Z,4,5,5


In [165]:
df_expanded["groundedness"] = pd.to_numeric(
    df_expanded["groundedness"], errors="coerce"
)


In [166]:
avg_by_dialogue = (
    df_expanded
    .groupby("dialogue_id")[["factual_accuracy", "helpfulness", "groundedness"]]
    .mean()
    .reset_index()
)


In [167]:
avg_by_dialogue

,dialogue_id,factual_accuracy,helpfulness,groundedness
0,1,3.250000,3.750000,3.00
1,2,5.000000,5.000000,5.00
2,3,4.800000,4.800000,5.00
3,4,4.333333,4.666667,5.00
4,5,5.000000,5.000000,5.00
5,6,4.250000,4.750000,4.50
6,7,4.500000,5.000000,4.75
7,8,5.000000,5.000000,5.00


In [168]:
#saving scores
avg_by_dialogue.to_csv("avg_by_dialogue_llm_judge.csv", index=False)

In [169]:
overall_avg = avg_by_dialogue[["factual_accuracy", "helpfulness", "groundedness"]].mean()
print(overall_avg)


factual_accuracy    4.516667
helpfulness         4.745833
groundedness        4.656250
dtype: float64


In [170]:
#saving average scores
with open("overall_avg_llm_judge.json", "w") as f:
    json.dump(overall_avg.to_dict(), f, indent=4)